# msg_parts

> The canonical LLM message model - `Msg`, `Part`, `ToolCall` - and its formatted text form

In [ ]:
#| default_exp msg_parts

`aidialog.msg_parts` holds the data structures every layer of the ecosystem shares: the canonical in-memory form of an LLM conversation (`Msg`, `Part`, `ToolCall`), the helpers that build them from plain content (`mk_msg`, `mk_msgs`), and the fenced-JSON text form that serializes tool calls and usage into markdown (`fmt2hist`, `hist2fmt`). It depends only on `fastcore`, so packages that need to speak the message format - dialog libraries, transcript tools, compaction - get it without depending on an LLM client. `fastllm` builds its chat clients on top of these same types.

In [ ]:
#| export
import base64, json
from json import dumps
from dataclasses import dataclass, field
from fastcore.utils import *
from fastcore.xtras import detect_mime

In [ ]:
#| hide
from fastcore.test import *
from fastcore.xml import Safe
from IPython.display import Markdown

## Part

Canonical atomic content unit for multimodal inputs/outputs.

**Why it exists**
- Providers represent content blocks differently (`input_text`, `text`, `inlineData`, `source`, etc).
- `Part` gives one stable internal shape so serialization/parsing logic can stay at provider boundaries.

**Design Notes**
- `type` encodes semantic modality (`text`, `input_image`, `input_file`, etc).
- `text` is for simple text payloads.
- `data` carries structured provider-agnostic payload details when text is insufficient.

**Connections**
- Wrapped by `Msg.content`.
- Produced/consumed by `highlevel` coercion, provider serializers in `clients`, and normalizers in `normalize`.
- Key enabler for model-only swapping without rewriting message construction code.

All four providers represent content parts differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Content is an array of typed parts: `{"type": "input_text", "text": "..."}`, `{"type": "input_image", "image_url": "..."}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "input_file", "file_data": "data:application/pdf;base64,...", "filename": "..."}`
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Content is an array of typed parts: `{"type": "text", "text": "..."}`, `{"type": "image_url", "image_url": {"url": "..."}}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "file", "file": {"file_data": "data:application/pdf;base64,...", "filename": "..."}}`
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Content blocks with `source` nesting: `{"type": "text", "text": "..."}`, `{"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": "..."}}`, `{"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": "..."}}`
- **[Gemini](https://ai.google.dev/api/generate-content)** — A `parts` array of `{text}` or `{inlineData: {mimeType, data}}` or `{fileData: {mimeType, fileUri}}`: `{"text": "..."}`, `{"inlineData": {"mimeType": "image/jpeg", "data": "..."}}`, `{"fileData": {"mimeType": "application/pdf", "fileUri": "..."}}`

> **Spec sources:** OpenAI Responses `Input{Text,Image,File}Content` (`specs/openai.with-code-samples.yml:68093–68444`), OpenAI Chat `ChatCompletionRequestMessageContentPart*` (`specs/openai.with-code-samples.yml:35185–35321`), Anthropic `Request{Text,Image,Document}Block` (`specs/anthropic.yml:12159–12458`), Gemini `Part`/`Blob`/`FileData` (`specs/gemini.json:189–387`).

`Part` canonicalizes these into a uniform `(type, text, data)` triple. The canonical types, their aliases, and expected input formats:

| Canonical `Part.type` | Aliases accepted | Expected input format | `Part.text` | `Part.data` |
|---|---|---|---|---|
| `"text"` | — | Plain string | `"the text"` | `None` |
| `"input_image"` | `image`, `image_url` | URL, data URL (`data:image/...;base64,...`), or provider-native `source`/`inlineData` in `data` | URL shorthand | `{"url": "..."}` or `{"image_url": "..."}` or provider-native payload |
| `"input_audio"` | `audio` | base64 data + format (OpenAI), URL/`fileUri` (Gemini). Not supported on Anthropic. | URL shorthand | `{"input_audio": {"data": "...", "format": "wav"}}` or `{"inlineData": {...}}` |
| `"input_video"` | `video`, `video_url` | URL or `fileUri`. Currently Gemini only; OpenAI maps to `input_file`. Not supported on Anthropic. | URL shorthand | `{"video_url": "..."}` or `{"fileData": {...}}` |
| `"input_file"` | `file`, `pdf`, `document` | data URL, URL, `file_id`, or provider-native `source` in `data` | URL shorthand | `{"file_data": "data:...;base64,...", "filename": "..."}` or `{"source": {...}}` |

**Provider support matrix:**

| Canonical type | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `text` | ✅ `input_text` | ✅ `text` | ✅ `text` | ✅ `text` |
| `input_image` | ✅ `input_image` | ✅ `image_url` | ✅ `image` + `source` | ✅ `inlineData` / `fileData` |
| `input_audio` | ✅ `input_audio` | ✅ `input_audio` | ❌ (raises `UnsupportedCapabilityError`) | ✅ `inlineData` / `fileData` |
| `input_video` | ✅ (mapped to `input_file`) | ✅ (mapped to `input_file`) | ❌ (raises `UnsupportedCapabilityError`) | ✅ `fileData` |
| `input_file` | ✅ `input_file` | ✅ `file` | ✅ `document` + `source` | ✅ `inlineData` / `fileData` |

**Escape hatch:** For any type, setting `Part.data["<provider>"]` (e.g. `Part.data["anthropic"]`) to a dict bypasses canonicalization and passes the payload directly to that provider (see `_provider_part`).

**Canonicalization design:** `Part.data` uses OpenAI-style key conventions as the single canonical input format. Users write one shape (e.g. `{"image_url": "..."}`, `{"file_data": "data:...;base64,..."}`, `{"input_audio": {"data": "...", "format": "wav"}}`), and the provider serializers (`_openai_responses_part`, `_anthropic_part`, `_gemini_part`) dispatch these into provider-native wire formats. This means users can swap models/providers without changing their `Part` construction code. `Part.text` serves as a URL shorthand fallback — e.g. `Part(type="input_image", text="https://example.com/img.png")` works when you just have a URL and no other metadata.

The key insight: `type` carries the **semantic modality**, not the provider-specific type name. Provider-specific payload details live in `data`, keeping `Part` itself provider-agnostic. This means downstream code can branch on `Part.type` without knowing which provider produced it.

In [ ]:
#| export
@dataclass
class Part:
    "A normalized content part."
    type: str
    text: str = None
    data: dict = None

In [ ]:
#| export
PartType = str_enum('PartType', 'text', 'thinking', 'refusal', 'tool_use', 'server_tool_result', 'tool_result',
                    'input_image', 'input_audio', 'input_video', 'input_file')

In [ ]:
#| export
def _trunc_strs(o, n=200):
    "Truncate str or dict"
    if not o: return o
    if isinstance(o,str) and len(o)>n: return o[:100]+'...'
    if isinstance(o,dict): return {k: (v[:100]+'...' if isinstance(v,str) and len(v)>n else v) for k,v in o.items()}
    return o

@patch
def _repr_markdown_(self: Part):
    body = _trunc_strs(self.text) if self.text else ''
    data = _trunc_strs(self.data)
    return f"""**Part** (`{self.type}`)

{body}

<details markdown='1'>

- data: `{data}`

</details>"""

In [ ]:
Part(PartType.text, 'Hello world!'*1000, data={'long':"10"*1000})

**Part** (`text`)

Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hell...

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

## Msg

Canonical conversation turn abstraction.

**Why it exists**
- Conversation structures vary across APIs (chat messages, response input items, content blocks).
- `Msg` lets the rest of fastllm reason in one conversation format.

**Design Notes**
- `role` captures turn semantics (`user`, `assistant`, `tool`, etc).
- `content` is a list of `Part` to support multimodal turns consistently.

**Connections**
- Primary input to `acompletion` and provider clients.
- Used by toolloop replay flows (`StreamSummary`/`Completion` -> `Msg` coercion in `highlevel`).

All three providers structure conversation messages differently:

- **[OpenAI Chat](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Messages are `{role, content}` objects. Roles: `system`, `developer`, `user`, `assistant`, `tool`. Content is a string or array of typed parts. Tool results use `role: "tool"` with `tool_call_id`.
- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Input items with `{role, content}`. Text parts become `input_text`/`output_text` depending on role. Tool results are `{type: "function_call_output", call_id, output}`.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Messages are `{role, content}` with roles `user` or `assistant` only. System prompt is a separate top-level parameter. Tool results are `tool_result` content blocks inside `role: "user"` messages.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Messages are `{role, parts}`. Roles: `user` or `model`. System prompt uses `system_instruction`. Tool results are `functionResponse` parts inside `role: "user"` messages.

> **Spec sources:** OpenAI `ChatCompletionRequest*Message` (`specs/openai.with-code-samples.yml:35022–35444`), Anthropic `InputMessage` + `RequestToolResultBlock` (`specs/anthropic.yml:11140–11159,12595–12652`), Gemini `Content` (`specs/gemini.json:171–188`).

`Msg` normalizes these into a canonical `(role, content, data)` triple:

| Canonical `Msg.role` | OpenAI Chat | OpenAI Responses | Anthropic | Gemini |
|---|---|---|---|---|
| `"system"` | `role: "system"` | `role: "system"` | Separate `system` param | `system_instruction` |
| `"user"` | `role: "user"` | `role: "user"` | `role: "user"` | `role: "user"` |
| `"assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "model"` |
| `"tool"` | `role: "tool"` + `tool_call_id` | `function_call_output` + `call_id` | `role: "user"` + `tool_result` block | `role: "user"` + `functionResponse` |

**`Msg.data` metadata:** Provider-agnostic metadata that doesn't fit `role`/`content`:
- `tool_calls` — list of tool call dicts (assistant messages)
- `tool_call_id` / `call_id` / `id` — tool call identifier (tool result messages)
- `name` — tool function name (tool result messages)
- `is_error` — whether the tool result is an error (Anthropic-specific, forwarded)

**Escape hatch:** Like `Part`, setting `Msg.data["<provider>"]` (e.g. `Msg.data["anthropic"]`) to a dict with `"role"` bypasses serialization entirely and passes the raw message to that provider.

In [ ]:
#| export
@dataclass
class Msg:
    "A normalized message."
    role: str
    content: List[Part]

    @property
    def text(self): return ''.join(p.text or '' for p in self.content if p.type == PartType.text)

    def _repr_markdown_(self):
        return f"""**Msg**

- role: `{self.role}`

<contents>

{'\n\n'.join(p._repr_markdown_() for p in self.content)}

</contents>"""

In [ ]:
Msg('user', content=[Part(PartType.text, 'Hello world!', data={'long':"10"*1000})]*3)

**Msg**

- role: `user`

<contents>

**Part** (`text`)

Hello world!

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

**Part** (`text`)

Hello world!

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

**Part** (`text`)

Hello world!

<details markdown='1'>

- data: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`

</details>

</contents>

## ToolCall

All four providers represent tool invocations differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Tool calls are flat output items: `{type: "function_call", call_id, name, arguments}` where `arguments` is a JSON **string**. Streamed via `response.function_call_arguments.delta` events.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Tool calls are nested: `{id, type: "function", function: {name, arguments}}` where `arguments` is a JSON **string**. Streamed in chunks via `tool_calls[i].function.arguments` deltas.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Tool calls are `tool_use` content blocks: `{id, name, input, type: "tool_use"}` where `input` is a parsed JSON object. Streamed via `input_json_delta` events.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Tool calls are `functionCall` parts: `{id, name, args}` where `args` is a parsed JSON object. Not chunked during streaming.

> **Spec sources:** OpenAI Responses `FunctionToolCall` (`specs/openai.with-code-samples.yml:44347–44389`), OpenAI Chat `ChatCompletionMessageToolCall` (`specs/openai.with-code-samples.yml:34869–34894`), Anthropic `ResponseToolUseBlock` (`specs/anthropic.yml:13563–13588`), Gemini `FunctionCall` (`specs/gemini.json:273–293`).

`ToolCall` canonicalizes these into a flat `(id, name, arguments)` triple where `arguments` is always a parsed `dict`:

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini | `ToolCall` |
|---|---|---|---|---|---|
| ID | `call_id` | `id` | `id` | `id` | `id` |
| Name | `name` | `function.name` | `name` | `name` | `name` |
| Args | `arguments` (JSON string) | `function.arguments` (JSON string) | `input` (object) | `args` (object) | `arguments` (dict) |

In [ ]:
#| export
@dataclass
class ToolCall:
    "Normalized tool call."
    id: str
    name: str
    arguments: dict = field(default_factory=dict)
    server: bool = False
    extra: dict = field(default_factory=dict)

In [ ]:
#| export
@patch
def _repr_markdown_(self: ToolCall):
    extra = _trunc_strs(self.extra)
    return f"""🔧 **{self.name}**(`{self.arguments}`)

<details markdown='1'>

- id: `{self.id}`
- server: `{self.server}`
- extra: `{extra}`

</details>"""

def display_list(l): 
    from IPython.display import Markdown, display
    display(Markdown('\n\n'.join(o._repr_markdown_() for o in l)))

In [ ]:
ToolCall(id='oxwvx1fm', name='simple_add', arguments={'b': 547982745, 'a': 5478954793}, server=False, extra={'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY76KxMeW9jpsnUvh6ZjPM5sDD7fAafF7cjdApNMsihPqIZBAZjAlFPcp1c/50MObH5f1q7hO7fgDS4iSJ3Q3FfbAYWnJ4nlA2peVMu/6WFcKZh1wcZCIuN6iFCj6nhH+6RKkaFRaM0b6XCmpti6qldSeZx+qtHmo+lzr1tct4Gz/CITDI7gRJ3qfLYV2u45jOhKzdd1t6gQ39XLJ93j0xd0AwpzcdZLbHWqwWJCQ43nNzhJ7IQTAWOSyPgKDnlAMHq2PTEoXBYkMBApCZ1x+HncBzt77kQrTTe7sWGVmD5boVnYAIFPFGXOULP5tDZ+nog+Fg8NV10vaFKlHVf+VDzFnVWxT259LN12ykGtBilfpTXiKCV12RAZwhuL7vXXHrsBGg5HNVImcXqgMvwf/rtQlJeop+9bEcAiU48hMFMzumOrCmmHD3HgxpYLW7T3vtDmbNdKCDqVtIwO4Rp5HE6GudRWmq8iC2UnyQglUXoXVnxIZW7eYYDsGAYrYgZ1A='})

🔧 **simple_add**(`{'b': 547982745, 'a': 5478954793}`)

<details markdown='1'>

- id: `oxwvx1fm`
- server: `False`
- extra: `{'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY...'}`

</details>

## Message utilities

`mk_tool_res_msg` pairs parallel tool calls with their results as one `tool`-role message, one `tool_result` part per call.

In [ ]:
#| export
def mk_tool_res_msg(tool_calls:list[ToolCall], results:list[str|list]):
    'A util to prepare parallel tool call with str or media list results'
    parts = []
    for tc,res in zip(tool_calls, results):
        data = dict(id=tc.id, name=tc.name, arguments=tc.arguments, server=tc.server)
        parts.append(Part(type=PartType.tool_result, text=res, data=data))
    return Msg(role="tool", content=parts)

`sys_text` and `part_txt` accept either bare strings or `Part`s, so call sites need not care which form they hold.

In [ ]:
#| export
def sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

def part_txt(p): return p.text if isinstance(p,Part) else p

Media referenced by URL needs a MIME type. `data_url` parses `data:` URLs, and `url_mime` guesses from the extension, falling back to sniffing the first bytes of the resource; the fetch imports `httpx` lazily, so aidialog's hard dependencies stay at fastcore alone. `MediaUrl` bundles a URL with its MIME type - the reference form for media passed by URL rather than downloaded.


In [ ]:
#| export
@flexicache(time_policy(24*3600))
def _fetch_url_partial(url, nbytes=512): 
    "Fetch remote media bytes, optionally only first `nbytes`."
    import httpx  # deliberately lazy: keeps aidialog's deps to fastcore alone (may re-base on fastcore.net later)
    try:
        with httpx.stream('GET', url, headers={'Range': f'bytes=0-{nbytes-1}'}, follow_redirects=True) as r:
            if r.status_code not in (200, 206): return
            return r.read()
    except (httpx.HTTPError, httpx.InvalidURL): return

In [ ]:
#| export
_ext_mime = {
    '.jpg':'image/jpeg', '.jpeg':'image/jpeg', '.png':'image/png', '.gif':'image/gif', '.webp':'image/webp',
    '.pdf':'application/pdf',
    '.mp3':'audio/mpeg', '.wav':'audio/wav', '.ogg':'audio/ogg', '.flac':'audio/flac', '.m4a':'audio/mp4',
    '.mp4':'video/mp4', '.mov':'video/quicktime', '.webm':'video/webm',
}

def data_url(url):
    "Parse data:mime;base64,data URL into (mime, b64_data), or None."
    if not isinstance(url, str) or not url.startswith('data:') or ',' not in url: return None
    header, body = url.split(',', 1)
    if ';base64' not in header or not body: return None
    return header[5:].split(';',1)[0].strip() or 'application/octet-stream', body

def url_mime(url, default='application/octet-stream'):
    "Guess mime from URL extension, and optional bytes fallback."
    if "youtube.com" in url or "youtu.be" in url: return "video/mp4"
    ext = '.' + url.rsplit('.', 1)[-1].split('?')[0].lower() if '.' in url.split('?')[0].split('/')[-1] else ''
    if (mime:=_ext_mime.get(ext)) is None: return detect_mime(_fetch_url_partial(url))
    return ifnone(mime, default)

In [ ]:
#| export
class MediaUrl(BasicRepr):
    "Direct URL media reference"
    def __init__(self, url, mime=None): self.url, self.mime = url, ifnone(mime, url_mime(url))

Content values become `Part`s through one dispatch: a str is text, bytes are sniffed with `detect_mime` and inlined as a base64 data URL, and a `MediaUrl` becomes a reference part. fastllm's `mk_msg` builds on this dispatch, adding provider concerns (`Completion` unwrapping, cache control) on its side of the boundary.

In [ ]:
#| export
def _mime2part_type(mime):
    "Map MIME string to canonical PartType"
    if mime.startswith('image/'): return PartType.input_image
    if mime.startswith('audio/'): return PartType.input_audio
    if mime.startswith('video/'): return PartType.input_video
    return PartType.input_file

def _bytes2content(data):
    "Convert bytes to fastllm canonical content"
    mtype = detect_mime(data)
    if not mtype: raise ValueError(f'Data must be a supported file type, got {data[:10]}')
    encoded = base64.b64encode(data).decode("utf-8")
    return Part(type=_mime2part_type(mtype), text=f'data:{mtype};base64,{encoded}')

def _url2content(o):
    "Convert MediaUrl to fastllm canonical content"
    mime = o.mime or url_mime(o.url)
    return Part(type=_mime2part_type(mime), text=o.url, data=dict(mime=mime))

def _mk_content(o):
    if isinstance(o, str):        return Part(type=PartType.text, text=o)
    elif isinstance(o, bytes):    return _bytes2content(o)
    elif isinstance(o, MediaUrl): return _url2content(o)
    return o

## The formatted text form

A conversation with tool calls can be rendered as one markdown string, with each call and its result serialized as a fenced JSON block whose info string is `json {.tool}` (and token usage as `json {.usage}`). This fixture is in the *legacy* envelope format that earlier fastllm releases shipped; `conv_tools` upgrades it to the fenced wire format, and is idempotent:

In [ ]:
fmt_outp = '''
I'll solve this step-by-step, using parallel calls where possible.

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "call": { "function": "simple_add", "arguments": { "a": 10, "b": 5 } },
  "result": "15",
  "server": false
}
```

</details>

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "call": { "function": "simple_add", "arguments": { "a": 2, "b": 1 } },
  "result": "3",
  "server": false
}
```

</details>

Now I need to multiply 15 * 3 before I can do the final division:

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "call": { "function": "multiply", "arguments": { "a": 15, "b": 3 } },
  "result": "45",
  "server": false
}
```

</details>

<details class='token-usage-details' markdown='1'><summary>Cache hit: 81.8% | Tokens: total=23,276 input=23,158 (+18,910 cached, 0 new) output=118 (reasoning 23)</summary>

`Usage(prompt_tokens=3, completion_tokens=10, total_tokens=13, raw={'input_tokens': 3, 'cache_creation_input_tokens': 2079, 'cache_read_input_tokens': 2070, 'cache_creation': {'ephemeral_5m_input_tokens': 2079, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 10, 'service_tier': 'standard', 'inference_geo': 'global'})`

</details>
'''

In [ ]:
#| export
tool_info = 'json {.tool}'     # fence info string of a tool block: {id, name, args, result} (+server; `error` reserved)
usage_info = 'json {.usage}'   # fence info string of a usage block: UsageStats fields

def parse_tools(s):
    "Split `s` into `(text, data)` segments: `data` is a parsed `{.tool}` block dict, `None` for the final segment"
    res, pos = [], 0
    for info,body,start,end in fenced_blocks(s):
        if info != tool_info: continue
        try: d = json.loads(body)
        except Exception: continue
        res.append((s[pos:start], d))
        pos = end
    return res + [(s[pos:], None)]

def strip_tools(s, tools=True, usage=True):
    "Remove `{.tool}` (and `{.usage}`) blocks from `s`"
    out, pos = [], 0
    for info,body,start,end in fenced_blocks(s):
        if not (tools and info == tool_info) and not (usage and info == usage_info): continue
        out.append(s[pos:start])
        pos = end
    out.append(s[pos:])
    return ''.join(out)

think_start,think_end = '<!--think_start-->','<!--think_end-->'
re_think = re.compile(rf'{re.escape(think_start)}.*?{re.escape(think_end)}\n?', re.DOTALL)

# Frozen legacy envelope recognition, used only by `conv_tools`. These are the
# exact patterns fastllm shipped for the released `<details markdown='1'>`
# envelopes plus the never-released `::: {.details}` spelling.
_lg_tool_tag = "<details class='tool-usage-details' markdown='1'>"
_lg_token_tag = "<details class='token-usage-details' markdown='1'>"
_lg_tool_attrs, _lg_token_attrs = "{.details .tool-usage-details}", "{.details .token-usage-details}"
_lg_tools = re.compile(
    fr"^(?:{_lg_tool_tag}\n*(?:<summary>(?P<summ1>.*?)</summary>\n*)?\n*```json\n+(?P<json1>.*?)\n+```\n+</details>"
    fr"|(?P<fence>:{{3,}}) {re.escape(_lg_tool_attrs)}\n+(?:## (?P<summ2>.*?)\n+)?```json\n+(?P<json2>.*?)\n+```\n+(?P=fence)$)",
    flags=re.DOTALL|re.MULTILINE)
_lg_token = re.compile(
    fr"^(?:{re.escape(_lg_token_tag)}\n*<summary>(?P<tsumm1>.*?)</summary>\n*\n*`(?P<trepr1>.*?)`\n*\n*</details>"
    fr"|(?P<tfence>:{{3,}}) {re.escape(_lg_token_attrs)}\n+## (?P<tsumm2>.*?)\n+`(?P<trepr2>.*?)`\n+(?P=tfence)$)\n?",
    flags=re.DOTALL|re.MULTILINE)

def conv_tools(s):
    "Convert legacy tool/usage envelopes in `s` (both historical spellings) to the fenced JSON wire format. Idempotent."
    def _tool(m):
        tj = m['json1'] if m['json1'] is not None else m['json2']
        try: d = json.loads(tj.strip())
        except Exception: return m[0]
        call = d.get('call') or {}
        res = dict(id=d.get('id'), name=call.get('function'), args=call.get('arguments') or {}, result=d.get('result'))
        if d.get('server'): res['server'] = True
        return fenced(dumps(res, indent=2, ensure_ascii=False), tool_info)
    def _tok(m):
        summ = m['tsumm1'] if m['tsumm1'] is not None else m['tsumm2']
        det = m['trepr1'] if m['trepr1'] is not None else m['trepr2']
        return fenced(dumps(dict(summary=summ, detail=det), ensure_ascii=False), usage_info)
    return _lg_token.sub(_tok, _lg_tools.sub(_tool, s))

In [ ]:
wire_outp = conv_tools(fmt_outp)
test_eq(conv_tools(wire_outp), wire_outp)
assert 'json {.tool}' in wire_outp and 'details' not in wire_outp
Markdown(wire_outp)


I'll solve this step-by-step, using parallel calls where possible.

```json {.tool}
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "name": "simple_add",
  "args": {
    "a": 10,
    "b": 5
  },
  "result": "15"
}
```

```json {.tool}
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "name": "simple_add",
  "args": {
    "a": 2,
    "b": 1
  },
  "result": "3"
}
```

Now I need to multiply 15 * 3 before I can do the final division:

```json {.tool}
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "name": "multiply",
  "args": {
    "a": 15,
    "b": 3
  },
  "result": "45"
}
```

```json {.usage}
{"summary": "Cache hit: 81.8% | Tokens: total=23,276 input=23,158 (+18,910 cached, 0 new) output=118 (reasoning 23)", "detail": "Usage(prompt_tokens=3, completion_tokens=10, total_tokens=13, raw={'input_tokens': 3, 'cache_creation_input_tokens': 2079, 'cache_read_input_tokens': 2070, 'cache_creation': {'ephemeral_5m_input_tokens': 2079, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 10, 'service_tier': 'standard', 'inference_geo': 'global'})"}
```

`parse_tools` splits the wire form into `(text, tool_dict)` segments - the parsing primitive `fmt2hist` builds on:

In [ ]:
segs = parse_tools(wire_outp)
test_eq(len(segs), 4)
[(txt.strip()[:40], d and d['name']) for txt,d in segs]

[("I'll solve this step-by-step, using para", 'simple_add'),
 ('', 'simple_add'),
 ('Now I need to multiply 15 * 3 before I c', 'multiply'),
 ('```json {.usage}\n{"summary": "Cache hit:', None)]

### Result fences

In [ ]:
#| export
_fence_back = '`````'
_result_re = re.compile(f'\n{_fence_back}result\n(.*?)\n{_fence_back}\n', re.DOTALL)


In [ ]:
#| export
def _mk_result_fence(output): return f"\n{_fence_back}result\n{output}\n{_fence_back}\n"

def _split_msg_on_fences(msg):
    "Split an assistant Msg on result fences, return list of Msgs"
    if msg.role != 'assistant': return [msg]
    if not _result_re.search(msg.text): return [msg]
    res, asst_parts, tool_parts = [], [], []
    for msg_part in msg.content:
        if msg_part.type == PartType.thinking: asst_parts.append(msg_part)
        elif msg_part.type == PartType.tool_use: tool_parts.append(msg_part)
        elif parts := _result_re.split(msg_part.text or ''):
            for i,p in enumerate(parts):
                if not p: continue
                if i % 2 == 0: res.append(Msg(role='assistant', content=asst_parts+[Part(type=PartType.text, text=p.strip())]))
                else:          res.append(Msg(role='user', content=[Part(type=PartType.text, text=_mk_result_fence(p))]))
    if tool_parts: res.append(Msg(role='assistant', content=tool_parts))
    return res

def _split_fence_msgs(msgs):
    "Split all assistant msgs on result fences for wire protocol"
    res = []
    for m in msgs: res.extend(_split_msg_on_fences(m))
    return res

In [ ]:
# No fence — unchanged
msg = Msg(role='assistant', content=[Part(PartType.text, 'Hello world')])
res = _split_msg_on_fences(msg)
test_eq(len(res), 1); test_eq(res[0].role, 'assistant')

# One result fence
msg = Msg(role='assistant', content=[Part(PartType.text, 'Let me calculate.\n`````py\n1+1\n`````\n\n`````result\n2\n`````\n\nDone.')])
res = _split_msg_on_fences(msg)
test_eq([m.role for m in res], ['assistant', 'user', 'assistant'])
test_eq('`````py\n1+1' in res[0].content[0].text, True)
test_eq('`````result\n2\n`````' in res[1].content[0].text, True)
test_eq('Done.' in res[2].content[0].text, True)

# Non-assistant — unchanged
msg = Msg(role='user', content=[Part(PartType.text, '`````result\n2\n`````')])
res = _split_msg_on_fences(msg)
test_eq([m.role for m in res], ['user'])

# Thinking first
msg = Msg(role='assistant', content=[
    Part(type=PartType.thinking, text='The user wants me to write an RNG function...', data={'citations': []}),
    Part(type=PartType.text, text='`````py\nimport random\n\ndef rng():\n    return random.random()\n\nprint(rng())\n`````\n`````result\n42\n`````\n', data={'citations': []})
])
res = _split_fence_msgs([msg])
test_eq([m.role for m in res], ['assistant', 'user'])
test_eq([p.type for p in res[0].content], ['thinking', 'text'])

# mixed code fence and tool call
msg = Msg(role='assistant', content=[
    Part(type=PartType.text, text='`````py\nimport random\n\ndef rng():\n    return random.random()\n\nprint(rng())\n`````\n`````result\n42\n`````\n', data={'citations': []}),
    Part(type=PartType.tool_use, text=None, data={'id': '4vy96hyd', 'name': 'python', 'arguments': {'code': 'import random\nprint(random.randint(1, 100))'}, 'server': False})])
res = _split_msg_on_fences(msg)
test_eq([m.role for m in res], ['assistant', 'user', 'assistant'])
test_eq([m.content[0].type for m in res], ['text', 'text', 'tool_use'])


### fmt2hist

In [ ]:
#| export
def _extract_tool_parts(d:dict):
    "Build (tool_use_part, tool_result_part) from a parsed `{.tool}` block"
    # Skip server tool calls in deserialization (round trip issues with Gemini/Anthropic)
    if not d or d.get('server') or d.get('id') is None: return None
    tu = Part(type=PartType.tool_use, text=None, data=dict(id=d['id'], name=d['name'], arguments=d.get('args') or {}))
    tr = Part(type=PartType.tool_result, text=str(d.get('result')), data={'id': d['id'], 'name': d['name']})
    return tu, tr

In [ ]:
#| export
def fmt2hist(outp:str)->list[Msg]:
    "Transform a formatted output string into fastllm canonical Msgs"
    if usage_info in outp: outp = strip_tools(outp, tools=False)
    if think_start in outp: outp = re_think.sub('', outp)
    if tool_info not in outp:
        msg = Msg(role='assistant', content=[Part(type=PartType.text, text=outp.strip() or '.')])
        return _split_msg_on_fences(msg)
    hist, asst_parts, tool_parts = [], [], []
    def flush():
        if tool_parts:
            hist.append(Msg(role='assistant', content=asst_parts.copy()))
            hist.append(Msg(role='tool',      content=tool_parts.copy()))
            asst_parts.clear()
            tool_parts.clear()
    for txt,d in parse_tools(outp.strip()):
        if txt and txt.strip():
            if tool_parts: flush()
            asst_parts.append(Part(type=PartType.text, text=txt.strip() or '.'))
        if d and (tp := _extract_tool_parts(d)):
            asst_parts.append(tp[0])
            tool_parts.append(tp[1])
    flush()
    if asst_parts: hist.append(Msg(role='assistant', content=asst_parts))
    if not hist: hist.append(Msg(role='assistant', content=[Part(type=PartType.text, text='.')]))
    result = []
    for msg in hist:
        if msg.role == 'assistant': result.extend(_split_msg_on_fences(msg))
        else: result.append(msg)
    if result[-1].role == 'tool': result.append(Msg(role='assistant', content=[Part(type=PartType.text, text='.')])) 
    return result

See how we can turn that one formatted output string back into a list of Msg:

In [ ]:
h = fmt2hist(wire_outp)
test_eq([m.role for m in h], ['assistant','tool','assistant','tool','assistant'])
h[1]

**Msg**

- role: `tool`

<contents>

**Part** (`tool_result`)

15

<details markdown='1'>

- data: `{'id': 'toolu_01KjnQH2Nsz2viQ7XYpLW3Ta', 'name': 'simple_add'}`

</details>

**Part** (`tool_result`)

3

<details markdown='1'>

- data: `{'id': 'toolu_01Koi2EZrGZsBbnQ13wuuvzY', 'name': 'simple_add'}`

</details>

</contents>

A tool response can be a string or a list of tool blocks (e.g., an image url block). To allow users to specify if a response should not be immediately stringified, we provide the `ToolResponse` datatype users can wrap their return statement in.

In [ ]:
#| export
@dataclass
class ToolResponse:
    content: list[str,str]

`StopResponse` and `FullResponse` are `str` subclasses that mark a string's handling downstream: a `StopResponse` tool result ends a tool loop, and a `FullResponse` must never be truncated. `_trunc_str` honors the latter (along with fastcore's `Safe` and `PrettyString`, checked by class name so no import is needed), and the `𝍁...𝍁` marker is the same contract for strings that crossed a serialization boundary:

In [ ]:
#| export
class StopResponse(str): pass
class FullResponse(str): pass

In [ ]:
#| export
def _trunc_str(s, mx=2000, skip=10, replace="TRUNCATED"):
    "Truncate `s` to `mx` chars max, adding `replace` if truncated; `mx=None` disables truncation"
    if mx is None or isinstance_str(s, ('FullResponse','Safe','PrettyString')): return s
    if not isinstance(s, str): s = str(s)
    s = type(s)(s.rstrip())
    if len(s)>2 and s[0]=='𝍁' and s[-1]=='𝍁':
        s = s[1:-1]
        if replace: return s
    if mx is None or len(s)<=mx: return s
    s = s[skip:mx-skip]
    ss = s.split(' ')
    if len(ss[-1])>150: ss[-1] = ss[-1][:5]
    s = ' '.join(ss)
    if skip: s = f"…{s}"
    s = f"{s}…"
    if replace: s = f"<{replace}>{s}</{replace}>"
    return s

In [ ]:
test_eq(_trunc_str('𝍁xxxxxxxxxx𝍁', mx=5), 'xxxxxxxxxx')
test_eq(_trunc_str(Safe('xxxxxxxxxx'), mx=5), 'xxxxxxxxxx')
test_eq(_trunc_str(FullResponse('xxxxxxxxxx'), mx=5), 'xxxxxxxxxx')
test_eq(_trunc_str('xxxxxxxxxx', mx=5, skip=0), '<TRUNCATED>xxxxx…</TRUNCATED>')
test_eq(_trunc_str('xxxxxxxxxx', mx=5, skip=1), '<TRUNCATED>…xxx…</TRUNCATED>')
test_eq(_trunc_str('xxxxxxxxxx', mx=None), 'xxxxxxxxxx')

In [ ]:
#| export
def _trunc_content(content, mx):
    "Truncate tool result content, respecting '_full' flag"
    if isinstance(content, dict) and '_full' in content and len(content)==1: return content['_full']
    return _trunc_str(content, mx=mx)

In [ ]:
#| export
def mk_tr_details(tr, mx=2000):
    "Create the `{.tool}` wire block for a tool call; `mx=None` disables truncation"
    args = {k:_trunc_str(v, mx=None if mx is None else mx*5) if isinstance(v, str) else v for k,v in tr.data['arguments'].items()}
    res = dict(id=tr.data['id'], name=tr.data['name'], args=args, result=_trunc_content(tr.text, mx=mx))
    if tr.data.get('server'): res['server'] = True
    return "\n\n" + fenced(dumps(res, indent=2, ensure_ascii=False), tool_info) + "\n\n"

`hist2fmt` is the inverse: it renders assistant/tool messages back into one formatted output string, with each tool call as the `<details>` block `fmt2hist` parses. This is how a captured conversation becomes an editable reply (e.g. a Solveit prompt output, or an llmsurgery dialog): text stays text, and every `tool_use`/`tool_result` pair folds into a details block carrying the call and its result. Results longer than `mx` are truncated by `mk_tr_details`, so the roundtrip is exact only within that limit; pass `mx=None` for an exact roundtrip with no truncation.

In [ ]:
#| export
def hist2fmt(msgs:list[Msg], mx=2000)->str:
    "Render assistant/tool `msgs` as one formatted output string, the inverse of `fmt2hist`"
    tus, out = {}, []
    for m in msgs:
        if m.role == 'assistant':
            for p in m.content:
                if p.type == PartType.text and p.text: out.append(p.text.strip())
                elif p.type == PartType.tool_use: tus[p.data['id']] = p
        elif m.role == 'tool':
            for p in m.content:
                if p.type != PartType.tool_result: continue
                tu = tus.get(p.data['id'])
                d = dict(p.data, arguments=tu.data.get('arguments', {}) if tu else {})
                out.append(mk_tr_details(Part(type=PartType.tool_result, text=p.text, data=d), mx=mx).strip())
        else: raise ValueError(f"hist2fmt renders assistant and tool messages only, got {m.role!r}")
    return '\n\n'.join(o for o in out if o)

In [ ]:
Markdown(hist2fmt(fmt2hist(wire_outp)))

I'll solve this step-by-step, using parallel calls where possible.

```json {.tool}
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "name": "simple_add",
  "args": {
    "a": 10,
    "b": 5
  },
  "result": "15"
}
```

```json {.tool}
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "name": "simple_add",
  "args": {
    "a": 2,
    "b": 1
  },
  "result": "3"
}
```

Now I need to multiply 15 * 3 before I can do the final division:

```json {.tool}
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "name": "multiply",
  "args": {
    "a": 15,
    "b": 3
  },
  "result": "45"
}
```

.

Parsing that rendering gives back exactly the messages we started from - `fmt2hist` and `hist2fmt` are inverses on the wire form.

In [ ]:
h2 = fmt2hist(hist2fmt(h))
test_eq(h2, h)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()